# Benchmark Runner — Full 25-Claim Evaluation

Runs all 25 benchmark claims through the two-vote evaluator and reports accuracy.

| Tier | Claims | Expected verdict |
|------|--------|------------------|
| WELL_SUPPORTED (WS) | 8 | SUPPORTED |
| CONTESTED (CT) | 7 | CONTESTED |
| OVERCLAIMED (OC) | 10 | UNSUPPORTED / INSUFFICIENT_EVIDENCE |

**Flow:**
1. Seed ChromaDB with papers covering all 25 claim domains (skip on re-runs)
2. `run()` calls `evaluate_claim()` on each claim, writes timestamped JSON + CSV to `benchmarks/results/`
3. Summary table, per-tier accuracy, and failed claims are displayed below

In [1]:
import sys, pathlib
# Kernel cwd is test/; parent is the project root
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print('Project root:', _root)

Project root: /Users/richardahn/projects/fibrosisLit


In [2]:
import logging, pandas as pd
from dotenv import load_dotenv
load_dotenv()

from benchmarks.benchmark_runner import run, seed_chromadb

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s — %(message)s')

## Step 1 — Seed ChromaDB (skip on re-runs)

Runs 13 MeSH-anchored PubMed queries covering all 25 claim domains and upserts results
into ChromaDB. Idempotent — safe to re-run but slow (~5 min). Skip if already populated.

In [3]:
# Skip this cell if ChromaDB is already populated from a previous run.
seed_chromadb(disease='ipf', max_results=50)

Seeding ChromaDB for disease='ipf' (13 queries)…


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

  seeded    4 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] SPP1 macrophage myofibroblast
  seeded   11 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] CTHRC1 fibroblast
  seeded    4 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] TGF-beta SMAD integrin
  seeded    5 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] bexotegrast integrin
  seeded    2 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] nerandomilast PDE4
  seeded   50 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] nintedanib clinical trial
  seeded    5 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] aberrant basaloid KRT17
  seeded    0 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] IL-13 autotaxin fibrosis
  seeded   39 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] LPA fibrosis
  seeded   50 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] bleomycin fibrosis
  seeded   50 papers | "idiopathic pulmonary fibrosis"[MeSH Terms] nintedanib bleomycin
  seeded    4 papers | "

## Step 2 — Run all 25 claims

Calls `evaluate_claim()` on each claim sequentially. Each call makes one Anthropic API
request. Exceptions are caught per claim so the run always completes all 25.
Results are written to `benchmarks/results/results_{timestamp}.json` and `.csv`.

In [4]:
# seed=False — ChromaDB already seeded in cell above
rows = run(disease='ipf')
print(f'\nReturned {len(rows)} rows.')

Running WS-01 (1/22)…
Running WS-02 (2/22)…
Running WS-03 (3/22)…
Running WS-04 (4/22)…
Running WS-05 (5/22)…
Running WS-06 (6/22)…
Running WS-07 (7/22)…
Running WS-08 (8/22)…
Running CT-01 (9/22)…
Running CT-02 (10/22)…
Running CT-03 (11/22)…
Running CT-04 (12/22)…
Running CT-05 (13/22)…
Running CT-06 (14/22)…
Running CT-07 (15/22)…
Running OC-01 (16/22)…
Running OC-02 (17/22)…
Running OC-03 (18/22)…
Running OC-04 (19/22)…
Running OC-05 (20/22)…
Running OC-06 (21/22)…
Running OC-07 (22/22)…

Results written to /Users/richardahn/projects/fibrosisLit/benchmarks/results/results_20260318T223604Z.json
CSV written to     /Users/richardahn/projects/fibrosisLit/benchmarks/results/results_20260318T223604Z.csv

Returned 22 rows.


## Step 3 — Full results table

In [5]:
df = pd.DataFrame(rows)
df['correct_bool'] = df['correct'].astype(bool)
df['correct'] = df['correct_bool'].map({True: '✓', False: '✗'})

display_cols = [
    'claim_id', 'tier', 'expected_verdict', 'actual_verdict',
    'verdict', 'verdict_confidence', 'correct',
    'prior_support_score', 'contested_flags',
]
pd.set_option('display.max_colwidth', 60)
display(df[display_cols].style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

claim_id,tier,expected_verdict,actual_verdict,verdict,verdict_confidence,correct,prior_support_score,contested_flags
WS-01,well_supported,SUPPORTED,SUPPORTED,SUPPORTED,HIGH,✓,0.850,
WS-02,well_supported,SUPPORTED,SUPPORTED,LOW_CONFIDENCE,LOW,✓,0.800,
WS-03,well_supported,SUPPORTED,SUPPORTED,SUPPORTED,HIGH,✓,1.000,
WS-04,well_supported,SUPPORTED,UNSUPPORTED,LOW_CONFIDENCE,LOW,✗,0.000,
WS-05,well_supported,SUPPORTED,SUPPORTED,SUPPORTED,HIGH,✓,0.800,
WS-06,well_supported,SUPPORTED,SUPPORTED,LOW_CONFIDENCE,LOW,✓,1.000,
WS-07,well_supported,SUPPORTED,SUPPORTED,SUPPORTED,HIGH,✓,0.750,
WS-08,well_supported,SUPPORTED,SUPPORTED,SUPPORTED,HIGH,✓,1.000,
CT-01,contested,CONTESTED,CONTESTED,CONTESTED,HIGH,✓,1.000,myofibroblast_reversibility
CT-02,contested,CONTESTED,CONTESTED,CONTESTED,HIGH,✓,0.000,macrophage_polarization


## Step 4 — Accuracy by tier

In [6]:
total   = len(df)
correct = df['correct_bool'].sum()
print(f'Overall: {correct}/{total} ({100*correct/total:.1f}%)\n')

for tier, g in df.groupby('tier'):
    n = len(g)
    c = g['correct_bool'].sum()
    print(f'  {tier:<14}: {c}/{n} ({100*c/n:.1f}%)')

Overall: 17/22 (77.3%)

  contested     : 6/7 (85.7%)
  overclaimed   : 4/7 (57.1%)
  well_supported: 7/8 (87.5%)


## Step 5 — Failed claims

In [7]:
failed = df[~df['correct_bool']]
if failed.empty:
    print('All claims passed.')
else:
    print(f'{len(failed)} failed claim(s):\n')
    fail_cols = [
        'claim_id', 'tier', 'expected_verdict', 'actual_verdict',
        'verdict', 'verdict_confidence', 'claim',
    ]
    display(failed[fail_cols].style.set_properties(**{'text-align': 'left'}).hide(axis='index'))

5 failed claim(s):



claim_id,tier,expected_verdict,actual_verdict,verdict,verdict_confidence,claim
WS-04,well_supported,SUPPORTED,UNSUPPORTED,LOW_CONFIDENCE,LOW,Nerandomilast (BI 1015550) met its primary endpoint in the Phase 3 FIBRONEER-IPF trial.
CT-06,contested,CONTESTED,UNSUPPORTED,UNSUPPORTED,HIGH,The bleomycin mouse model faithfully recapitulates IPF disease progression.
OC-02,overclaimed,INSUFFICIENT_EVIDENCE,CONTESTED,LOW_CONFIDENCE,LOW,M-CSF/CSF1R blockade is a validated therapeutic strategy for IPF based on macrophage biology.
OC-04,overclaimed,UNSUPPORTED,CONTESTED,LOW_CONFIDENCE,LOW,Anti-IL-13 therapy slows IPF progression.
OC-05,overclaimed,UNSUPPORTED,CONTESTED,CONTESTED,HIGH,Fibrosis resolution programs demonstrated in liver are directly applicable to IPF treatment.
